# Task 1: Data Exploration and Schema Review
## Forecasting Financial Inclusion in Ethiopia

**Objective**
- Understand the unified dataset schema
- Validate record types and relationships
- Prepare the ground for data enrichment

**Dataset**
- ethiopia_fi_unified_data.csv
- reference_codes.csv

**Analyst**
- Mulatie Kindie  
- Selam Analytics (Project Simulation)


# Import Libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)


# Load Raw Data

In [2]:
DATA_PATH = "../data/raw/"

fi_df = pd.read_csv(DATA_PATH + "ethiopia_fi_unified_data.csv")
ref_df = pd.read_csv(DATA_PATH + "reference_codes.csv")

print("Financial Inclusion Dataset Shape:", fi_df.shape)
print("Reference Codes Shape:", ref_df.shape)

Financial Inclusion Dataset Shape: (43, 1)
Reference Codes Shape: (71, 1)


# Preview the Unified Schema

In [3]:
fi_df.head()

,record_id\trecord_type\tcategory\tpillar\tindicator\tindicator_code\tindicator_direction\tvalue_numeric\tvalue_text\tvalue_type\tunit\tobservation_date\tperiod_start\tperiod_end\tfiscal_year\tgender\tlocation\tregion\tsource_name\tsource_type\tsource_url\tconfidence\trelated_indicator\trelationship_type\timpact_direction\timpact_magnitude\timpact_estimate\tlag_months\tevidence_basis\tcomparable_country\tcollected_by\tcollection_date\toriginal_text\tnotes\t\t
0,REC_0001\tobservation\t\tACCESS\tAccount Owner...
1,REC_0002\tobservation\t\tACCESS\tAccount Owner...
2,REC_0003\tobservation\t\tACCESS\tAccount Owner...
3,REC_0004\tobservation\t\tACCESS\tAccount Owner...
4,REC_0005\tobservation\t\tACCESS\tAccount Owner...


# explicitly inspect structure

In [4]:
fi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43 entries, 0 to 42
Data columns (total 1 columns):
 #   Column                                                                                                                                                                                                                                                                                                                                                                                                                                       Non-Null Count  Dtype 
---  ------                                                                                                                                                                                                                                                                                                                                                                                                                                       -------

# Understand the Unified Schema

In [5]:
fi_df.columns.tolist()

['record_id\trecord_type\tcategory\tpillar\tindicator\tindicator_code\tindicator_direction\tvalue_numeric\tvalue_text\tvalue_type\tunit\tobservation_date\tperiod_start\tperiod_end\tfiscal_year\tgender\tlocation\tregion\tsource_name\tsource_type\tsource_url\tconfidence\trelated_indicator\trelationship_type\timpact_direction\timpact_magnitude\timpact_estimate\tlag_months\tevidence_basis\tcomparable_country\tcollected_by\tcollection_date\toriginal_text\tnotes\t\t']

# Record Type Distribution

In [7]:
# Re-read with tab separator and fallback to splitting if needed
fi_df = pd.read_csv(DATA_PATH + "ethiopia_fi_unified_data.csv", sep='\t', engine='python')

if fi_df.shape[1] == 1:
    col = fi_df.columns[0]
    split_df = fi_df[col].astype(str).str.split('\t', expand=True)
    split_df.columns = split_df.iloc[0].str.strip()
    fi_df = split_df[1:].reset_index(drop=True)

# Now this will work
fi_df["record_type"].value_counts()

record_type
observation    30
event          10
target          3
Name: count, dtype: int64

The dataset follows a **long-format unified schema**, where:
- `record_type` determines semantic meaning
- All records share identical columns
- This design avoids structural bias

# Pillar vs Record Type Logic

In [8]:
pd.crosstab(fi_df["record_type"], fi_df["pillar"])

pillar,ACCESS,AFFORDABILITY,GENDER,USAGE
record_type,,,,
observation,14,1,4,11
target,2,0,1,0


# Explore Observations

In [9]:
obs_df = fi_df[fi_df["record_type"] == "observation"]

obs_df[[
    "pillar",
    "indicator",
    "indicator_code",
    "value_numeric",
    "observation_date",
    "source_name",
    "confidence"
]].head(10)


,pillar,indicator,indicator_code,value_numeric,observation_date,source_name,confidence
0,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,22.00,2014-12-31,Global Findex 2014,high
1,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,35.00,2017-12-31,Global Findex 2017,high
2,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,46.00,2021-12-31,Global Findex 2021,high
3,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,56.00,2021-12-31,Global Findex 2021,high
4,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,36.00,2021-12-31,Global Findex 2021,high
5,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,49.00,2024-11-29,Global Findex 2024,high
6,ACCESS,Mobile Money Account Rate,ACC_MM_ACCOUNT,4.70,2021-12-31,Global Findex 2021,high
7,ACCESS,Mobile Money Account Rate,ACC_MM_ACCOUNT,9.45,2024-11-29,Global Findex 2024,high
8,ACCESS,4G Population Coverage,ACC_4G_COV,37.50,2023-06-30,Ethio Telecom LEAD Report,high
9,ACCESS,4G Population Coverage,ACC_4G_COV,70.80,2025-06-30,Ethio Telecom LEAD Report,high


# Temporal Coverage of Observations

In [10]:
obs_df["observation_date"] = pd.to_datetime(obs_df["observation_date"])

obs_df["observation_date"].describe()

C:\Users\mulat\AppData\Local\Temp\ipykernel_15768\1079488955.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  obs_df["observation_date"] = pd.to_datetime(obs_df["observation_date"])


count                     30
mean     2024-01-12 08:48:00
min      2014-12-31 00:00:00
25%      2023-10-01 06:00:00
50%      2024-12-31 00:00:00
75%      2025-06-30 00:00:00
max      2025-12-31 00:00:00
Name: observation_date, dtype: object

In [11]:
obs_df.groupby(obs_df["observation_date"].dt.year)["indicator_code"].count()

observation_date
2014     1
2017     1
2021     5
2023     1
2024    11
2025    11
Name: indicator_code, dtype: int64

# Unique Indicators & Coverage

In [12]:
indicator_coverage = (
    obs_df.groupby("indicator_code")["observation_date"]
    .agg(["count", "min", "max"])
    .sort_values("count", ascending=False)
)

indicator_coverage

,count,min,max
indicator_code,,,
ACC_OWNERSHIP,6,2014-12-31,2024-11-29
ACC_FAYDA,3,2024-08-15,2025-05-15
ACC_4G_COV,2,2023-06-30,2025-06-30
ACC_MM_ACCOUNT,2,2021-12-31,2024-11-29
GEN_GAP_ACC,2,2021-12-31,2024-11-29
USG_P2P_COUNT,2,2024-07-07,2025-07-07
ACC_MOBILE_PEN,1,2025-12-31,2025-12-31
GEN_GAP_MOBILE,1,2024-12-31,2024-12-31
GEN_MM_SHARE,1,2024-12-31,2024-12-31


Key observations:
- Global Findex indicators have low frequency but high credibility
- Infrastructure and operator metrics help fill temporal gaps
- This justifies enrichment using proxy indicators

# Explore Events

In [14]:
events_df = fi_df[fi_df["record_type"] == "event"]

events_df[[
    "record_id",
    "indicator",
    "category",
    "observation_date",
    "source_name"
]].sort_values("observation_date")

,record_id,indicator,category,observation_date,source_name
33,EVT_0001,Telebirr Launch,product_launch,2021-05-17,Ethio Telecom
41,EVT_0009,NFIS-II Strategy Launch,policy,2021-09-01,NBE
34,EVT_0002,Safaricom Ethiopia Commercial Launch,market_entry,2022-08-01,News
35,EVT_0003,M-Pesa Ethiopia Launch,product_launch,2023-08-01,Safaricom
36,EVT_0004,Fayda Digital ID Program Rollout,infrastructure,2024-01-01,NIDP
37,EVT_0005,Foreign Exchange Liberalization,policy,2024-07-29,NBE
38,EVT_0006,P2P Transaction Count Surpasses ATM,milestone,2024-10-01,EthSwitch
39,EVT_0007,M-Pesa EthSwitch Integration,partnership,2025-10-27,EthSwitch
42,EVT_0010,Safaricom Ethiopia Price Increase,pricing,2025-12-15,News
40,EVT_0008,EthioPay Instant Payment System Launch,infrastructure,2025-12-18,NBE/EthSwitch


# Event Categories

In [15]:
events_df["category"].value_counts()

category
product_launch    2
infrastructure    2
policy            2
market_entry      1
milestone         1
partnership       1
pricing           1
Name: count, dtype: int64

# Impact Links

In [17]:
impact_df = fi_df[fi_df["record_type"] == "impact_link"]

impact_df = fi_df[fi_df["record_type"] == "impact_link"]

cols = [
    "parent_id",
    "pillar",
    "related_indicator",
    "impact_direction",
    "impact_magnitude",
    "lag_months",
    "evidence_basis"
]

existing_cols = [c for c in cols if c in impact_df.columns]
impact_df[existing_cols]

,pillar,related_indicator,impact_direction,impact_magnitude,lag_months,evidence_basis


# Join Events to Impact Links

In [19]:
# Safely merge impact links to events using available columns.
# events_df uses 'record_id', 'indicator' (holds event name) and 'observation_date' (event date).
# impact_df may not contain 'parent_id' — handle that gracefully.

if "parent_id" in impact_df.columns:
    right_cols = [c for c in ["record_id", "indicator", "observation_date", "category"] if c in events_df.columns]
    impact_joined = impact_df.merge(
        events_df[right_cols],
        left_on="parent_id",
        right_on="record_id",
        how="left"
    ).rename(columns={"indicator": "event_name", "observation_date": "event_date"})
else:
    # no linking column present — return impact_df with placeholder event columns
    impact_joined = impact_df.copy()
    impact_joined["event_name"] = np.nan
    impact_joined["event_date"] = pd.NaT
    if "category" not in impact_joined.columns:
        impact_joined["category"] = np.nan

cols_to_show = [
    "event_name",
    "category",
    "pillar",
    "related_indicator",
    "impact_direction",
    "impact_magnitude",
    "lag_months"
]
cols_to_show = [c for c in cols_to_show if c in impact_joined.columns]
impact_joined[cols_to_show]

,event_name,category,pillar,related_indicator,impact_direction,impact_magnitude,lag_months


# Reference Codes Validation

In [21]:
# Normalize reference codes table (handle tab-separated single-column read)
if ref_df.shape[1] == 1:
    col = ref_df.columns[0]
    split_df = ref_df[col].astype(str).str.split('\t', expand=True)
    # Use first row as header if it looks like one
    header = split_df.iloc[0].astype(str).str.strip()
    split_df.columns = header
    ref_df = split_df[1:].reset_index(drop=True)

# Drop any entirely-empty or unnamed columns
ref_df = ref_df.loc[:, [c for c in ref_df.columns if str(c).strip() != "" and ref_df[c].notna().any()]]

# Normalize column names
ref_df.columns = ref_df.columns.str.strip().str.lower().str.replace(r'\s+', '_', regex=True)

# Attempt to show counts for likely code-type columns
for candidate in ["code_type", "field", "code"]:
    if candidate in ref_df.columns:
        print(ref_df[candidate].value_counts())
        break
else:
    print("Available columns:", ref_df.columns.tolist())

Available columns: ['record_type', 'observation', 'actual_measured_value_from_a_source', 'all']


In [23]:
ref_df[ref_df["record_type"] == "pillar"]

,record_type,observation,actual_measured_value_from_a_source,all
15,pillar,ACCESS,Can people reach services? Coverage devices ac...,observation/target/impact_link
16,pillar,USAGE,Are people actively using? Transactions active...,observation/target/impact_link
17,pillar,QUALITY,Do services work? Success rates uptime,observation/target/impact_link
18,pillar,AFFORDABILITY,Can people afford it? Costs relative to income,observation/target/impact_link
19,pillar,TRUST,Do people trust it? Complaints fraud,observation/target/impact_link
20,pillar,DEPTH,Beyond payments? Savings credit insurance,observation/target/impact_link
21,pillar,GENDER,Gender gaps across dimensions,observation/target/impact_link


# Initial Data Quality Checks

In [24]:
fi_df["confidence"].value_counts(dropna=False)

confidence
high      40
NaN       16
medium     3
Name: count, dtype: int64

In [25]:
fi_df.isnull().mean().sort_values(ascending=False)

impact_estimate        1.000000
lag_months             1.000000
Unnamed: 35            1.000000
region                 1.000000
evidence_basis         1.000000
relationship_type      1.000000
related_indicator      1.000000
Unnamed: 34            1.000000
impact_direction       1.000000
impact_magnitude       1.000000
notes                  1.000000
category               0.830508
period_start           0.830508
collection_date        0.830508
period_end             0.830508
value_text             0.830508
source_url             0.474576
original_text          0.440678
pillar                 0.440678
indicator_direction    0.440678
value_numeric          0.440678
unit                   0.440678
record_id              0.271186
record_type            0.271186
source_type            0.271186
indicator_code         0.271186
value_type             0.271186
observation_date       0.271186
indicator              0.271186
source_name            0.271186
location               0.271186
gender  

# Summary

## Task 1 — Schema Review Summary

**Confirmed**
- Unified long-format schema
- Clean separation of events and impacts
- Sparse but high-quality outcome data

**Key Challenges**
- Limited time points for Findex indicators
- Heavy reliance on modeled event impacts
- Need for proxy indicators to improve forecasting

**Next Step**
- Enrich dataset with:
  - Infrastructure indicators
  - Usage proxies
  - Additional policy and market events